# 06 - Output Validation & Auto-Correction
**Module:** `src/utils/validator.py`

## What this module does
Acts as the safety net between the Groq API and our data layer.
Every raw LLM response passes through this module before anything
gets saved to disk.

## The 4-step validation pipeline
1. **Strip code fences** — remove ```json wrappers the model adds
2. **Parse JSON safely** — catch malformed JSON with a clear error
3. **Validate required fields** — check all fields are present
4. **Run guardrail checks** — embed post-generation flags into the lesson

## Why this matters
LLMs are non-deterministic. Even with a perfect prompt, the model
will occasionally return malformed output. This module means the
generator never crashes silently — every failure is caught, described,
and either fixed or reported clearly.

## What we'll explore
1. Code fence stripping
2. Safe JSON parsing — valid and invalid cases
3. Required field validation
4. The auto-correct for the practice dict bug
5. The full pipeline on a clean and a broken response

In [1]:
import sys
sys.path.insert(0, '..')

from src.utils.validator import (
    strip_code_fences,
    parse_json_safely,
    autocorrect_practice,
    validate_required_fields,
    validate_against_schema,
    validate_llm_output,
    ValidationError,
)

import json


## Part 1: Stripping Code Fences
Even when explicitly told not to, LLMs frequently wrap JSON
in markdown code fences. This step removes them silently.

In [2]:
# All the fence patterns the model might produce
examples = {
    "Clean JSON":           '{"lesson_id": "L-K2-SPK-001"}',
    "With ```json fence":   '```json\n{"lesson_id": "L-K2-SPK-001"}\n```',
    "With ``` fence":       '```\n{"lesson_id": "L-K2-SPK-001"}\n```',
    "With whitespace":      '  \n```json\n{"lesson_id": "L-K2-SPK-001"}\n```\n  ',
}

for label, raw in examples.items():
    cleaned = strip_code_fences(raw)
    print(f"{label}:")
    print(f"  Input  : {repr(raw[:50])}...")
    print(f"  Output : {repr(cleaned)}")
    print()

Clean JSON:
  Input  : '{"lesson_id": "L-K2-SPK-001"}'...
  Output : '{"lesson_id": "L-K2-SPK-001"}'

With ```json fence:
  Input  : '```json\n{"lesson_id": "L-K2-SPK-001"}\n```'...
  Output : '{"lesson_id": "L-K2-SPK-001"}'

With ``` fence:
  Input  : '```\n{"lesson_id": "L-K2-SPK-001"}\n```'...
  Output : '{"lesson_id": "L-K2-SPK-001"}'

With whitespace:
  Input  : '  \n```json\n{"lesson_id": "L-K2-SPK-001"}\n```\n  '...
  Output : '{"lesson_id": "L-K2-SPK-001"}'



## Part 2: Safe JSON Parsing
`parse_json_safely()` wraps `json.loads()` with a descriptive error.
Let's see what happens with valid and invalid JSON.

In [4]:
# Valid JSON
valid = '{"lesson_id": "L-K2-SPK-001", "metadata": {"grade_band": "K-2"}}'
result = parse_json_safely(valid)
print("Valid JSON parsed successfully!")
print("Result type:", type(result).__name__)
print("Keys:", list(result.keys()))

Valid JSON parsed successfully!
Result type: dict
Keys: ['lesson_id', 'metadata']


In [5]:
# Malformed JSON — missing closing brace
print("Testing malformed JSON:\n")
bad_examples = [
    ('Missing closing brace', '{"lesson_id": "L-K2-SPK-001"'),
    ('Plain text',            'Here is your lesson: blah blah'),
    ('Empty string',          ''),
    ('Truncated mid-field',   '{"lesson_id": "L-K2-SP'),
]

for label, bad_json in bad_examples:
    try:
        parse_json_safely(bad_json)
        print(f"  {label}: FAIL — should have raised ❌")
    except ValidationError as e:
        print(f"  {label}: Caught ValidationError ✅")

Testing malformed JSON:

  Missing closing brace: Caught ValidationError ✅
  Plain text: Caught ValidationError ✅
  Empty string: Caught ValidationError ✅
  Truncated mid-field: Caught ValidationError ✅


In [15]:
parse_json_safely('{"lesson_id": "L-K2-SP')

ValidationError: [validator] LLM returned invalid JSON.
JSON error: Unterminated string starting at: line 1 column 15 (char 14)
Raw output preview:
{"lesson_id": "L-K2-SP

## Part 3: The Practice Dict Auto-Correction
This is a real bug we encountered during development.
The model returned `practice` as a dict `{}` instead of a list `[]`.

Rather than failing, `autocorrect_practice()` detects and fixes it.
This makes the pipeline robust to a known, repeatable LLM mistake.

In [16]:
# Simulate the exact bug we encountered
lesson_with_dict_practice = {
    "lesson_id": "L-35-SPK-TEST",
    "lesson_flow": {
        "practice": {
            "P1": {
                "prompt_id": "P1",
                "type": "supported",
                "text": "Tell me about space.",
                "scaffold": "Start with: Space is..."
            },
            "P2": {
                "prompt_id": "P2",
                "type": "independent",
                "text": "Now describe a planet.",
                "scaffold": None
            }
        }
    }
}

print("Before autocorrect:")
print(f"  practice type: {type(lesson_with_dict_practice['lesson_flow']['practice']).__name__}")
print(f"  practice value: {lesson_with_dict_practice['lesson_flow']['practice']}\n")

corrected = autocorrect_practice(lesson_with_dict_practice)

print("\nAfter autocorrect:")
print(f"  practice type: {type(corrected['lesson_flow']['practice']).__name__}")
print(f"  practice length: {len(corrected['lesson_flow']['practice'])} prompts")
for p in corrected["lesson_flow"]["practice"]:
    print(f"  - {p['prompt_id']}: {p['text']}")

Before autocorrect:
  practice type: dict
  practice value: {'P1': {'prompt_id': 'P1', 'type': 'supported', 'text': 'Tell me about space.', 'scaffold': 'Start with: Space is...'}, 'P2': {'prompt_id': 'P2', 'type': 'independent', 'text': 'Now describe a planet.', 'scaffold': None}}

[validator] ⚠️  practice was a dict — auto-correcting to list...
[validator] Auto-corrected practice → 2 prompts ✅

After autocorrect:
  practice type: list
  practice length: 2 prompts
  - P1: Tell me about space.
  - P2: Now describe a planet.


## Part 4: Required Field Validation
If the LLM omits a required field, we catch it here with
a clear message -- not a cryptic KeyError somewhere downstream.

In [17]:
# Build a minimal valid lesson for testing
valid_lesson = {
    "lesson_id": "L-K2-SPK-TEST",
    "metadata": {
        "grade_band": "K-2",
        "ela_domain": "Speaking",
        "lesson_type": "Story Retell",
        "theme": "Animals",
        "primary_skill": "Retell a story in sequence",
        "voice_markers": ["Speaking Rate"],
        "estimated_duration_minutes": 6,
        "ccss_anchor": "CCSS.ELA-Literacy.SL.K.4"
    },
    "lesson_flow": {
        "hook":    {"duration_seconds": 60, "content": "A turtle finds an acorn."},
        "model":   {"duration_seconds": 80, "content": "Listen...",
                    "skill_named_explicitly": "Today we practice retelling."},
        "practice": [{"prompt_id": "P1", "type": "supported",
                      "text": "Tell me the beginning.", "scaffold": "Start with: First..."}],
        "reflect": {"duration_seconds": 45, "voice_marker_focus": "Speaking Rate",
                    "positive_signal": "Steady.", "growth_signal": "Pause more."}
    },
    "guardrail_flags": {}
}

# Valid lesson passes
validate_required_fields(valid_lesson)
print("Valid lesson passed field validation ✅")

Valid lesson passed field validation ✅


In [18]:
import copy

# Test missing fields one at a time
missing_field_tests = [
    ("lesson_id",           lambda d: d.pop("lesson_id")),
    ("metadata.primary_skill", lambda d: d["metadata"].pop("primary_skill")),
    ("lesson_flow.reflect", lambda d: d["lesson_flow"].pop("reflect")),
    ("lesson_flow.practice empty", lambda d: d["lesson_flow"].update({"practice": []})),
]

for label, mutate in missing_field_tests:
    test = copy.deepcopy(valid_lesson)
    mutate(test)
    try:
        validate_required_fields(test)
        print(f"  {label}: FAIL — should have raised ❌")
    except ValidationError as e:
        print(f"  Missing '{label}': Caught ValidationError ✅")
        print(f"    → {str(e)}")
    print()

  Missing 'lesson_id': Caught ValidationError ✅
    → [validator] Missing top-level fields: ['lesson_id']

  Missing 'metadata.primary_skill': Caught ValidationError ✅
    → [validator] Missing metadata fields: ['primary_skill']

  Missing 'lesson_flow.reflect': Caught ValidationError ✅
    → [validator] Missing lesson_flow fields: ['reflect']

  Missing 'lesson_flow.practice empty': Caught ValidationError ✅
    → [validator] lesson_flow.practice must be a non-empty list. Got: list



## Part 5: The Full Pipeline
`validate_llm_output()` runs all four steps in sequence.
Let's run it on three scenarios:
1. A clean response — should pass fully
2. A code-fenced response — should be cleaned and pass
3. A response with a content flag — should pass but record the flag

In [19]:
# Scenario 1: Clean response
print("=== SCENARIO 1: Clean response ===\n")
result = validate_llm_output(json.dumps(valid_lesson))
print(f"\nResult lesson_id: {result['lesson_id']}")
print("Guardrail flags:")
for check, flag in result["guardrail_flags"].items():
    icon = "✅" if flag["status"] == "pass" else "⚠️"
    print(f"  {icon} {check}: {flag['status']}")

=== SCENARIO 1: Clean response ===

[validator] Starting validation pipeline...
[validator] Step 1 — Code fences stripped ✅
[validator] Step 2 — JSON parsed successfully ✅
[validator] Step 3 — Required fields present ✅
[checks] single_skill_check: ✅ [PASS] Single skill confirmed: 'Retell a story in sequence'
[checks] vocabulary_ceiling_check: ✅ [PASS] All prompts within K-2 vocabulary ceiling (30 words).
[checks] cognitive_load_check: ✅ [PASS] Cognitive load check passed for grade band 'K-2'.
[checks] cultural_bias_check: ✅ [PASS] No cultural bias terms detected.
[validator] Step 4 — Guardrail checks complete ✅
[validator] Validation passed for lesson: L-K2-SPK-TEST

Result lesson_id: L-K2-SPK-TEST
Guardrail flags:
  ✅ single_skill_check: pass
  ✅ vocabulary_ceiling_check: pass
  ✅ cognitive_load_check: pass
  ✅ cultural_bias_check: pass


In [20]:
# Scenario 2: Code-fenced response — fences stripped automatically
print("=== SCENARIO 2: Code-fenced response ===\n")
fenced = "```json\n" + json.dumps(valid_lesson) + "\n```"
result2 = validate_llm_output(fenced)
print(f"\nResult lesson_id: {result2['lesson_id']} ✅")

=== SCENARIO 2: Code-fenced response ===

[validator] Starting validation pipeline...
[validator] Step 1 — Code fences stripped ✅
[validator] Step 2 — JSON parsed successfully ✅
[validator] Step 3 — Required fields present ✅
[checks] single_skill_check: ✅ [PASS] Single skill confirmed: 'Retell a story in sequence'
[checks] vocabulary_ceiling_check: ✅ [PASS] All prompts within K-2 vocabulary ceiling (30 words).
[checks] cognitive_load_check: ✅ [PASS] Cognitive load check passed for grade band 'K-2'.
[checks] cultural_bias_check: ✅ [PASS] No cultural bias terms detected.
[validator] Step 4 — Guardrail checks complete ✅
[validator] Validation passed for lesson: L-K2-SPK-TEST

Result lesson_id: L-K2-SPK-TEST ✅


In [21]:
# Scenario 3: Culturally biased content — passes but flags
print("=== SCENARIO 3: Content flag — passes but records warning ===\n")
biased = copy.deepcopy(valid_lesson)
biased["lesson_flow"]["hook"]["content"] = (
    "Every Thanksgiving, American families gather to share stories."
)
result3 = validate_llm_output(json.dumps(biased))
print(f"\nResult lesson_id: {result3['lesson_id']}")
print(f"cultural_bias_check: {result3['guardrail_flags']['cultural_bias_check']['status']}")
print(f"message: {result3['guardrail_flags']['cultural_bias_check']['message']}")

=== SCENARIO 3: Content flag — passes but records warning ===

[validator] Starting validation pipeline...
[validator] Step 1 — Code fences stripped ✅
[validator] Step 2 — JSON parsed successfully ✅
[validator] Step 3 — Required fields present ✅
[checks] single_skill_check: ✅ [PASS] Single skill confirmed: 'Retell a story in sequence'
[checks] vocabulary_ceiling_check: ✅ [PASS] All prompts within K-2 vocabulary ceiling (30 words).
[checks] cognitive_load_check: ✅ [PASS] Cognitive load check passed for grade band 'K-2'.
[checks] cultural_bias_check: ⚠️ [FLAG] Potential cultural specificity detected: ['thanksgiving', 'american']. Review whether non-US/non-Western students would have sufficient context. Consider adding a brief explainer or choosing a more globally neutral reference.
[validator] Step 4 — Guardrail checks complete ✅
[validator] Validation passed for lesson: L-K2-SPK-TEST

Result lesson_id: L-K2-SPK-TEST
cultural_bias_check: flag
message: Potential cultural specificity detec

In [23]:
# Scenario 4: Completely broken response — raises ValidationError
print("=== SCENARIO 4: Broken response — raises ValidationError ===\n")
try:
    validate_llm_output("sdfdsfsdfrequest.")
    print("FAIL ❌")
except ValidationError as e:
    print("Caught ValidationError ✅")
    print(f"\n{str(e)[:200]}...")

=== SCENARIO 4: Broken response — raises ValidationError ===

[validator] Starting validation pipeline...
[validator] Step 1 — Code fences stripped ✅
Caught ValidationError ✅

[validator] LLM returned invalid JSON.
JSON error: Expecting value: line 1 column 1 (char 0)
Raw output preview:
sdfdsfsdfrequest....


## Summary

| Step | What it catches | Raises or flags? |
|---|---|---|
| `strip_code_fences` | Markdown wrappers around JSON | Fixes silently |
| `parse_json_safely` | Malformed or non-JSON responses | Raises `ValidationError` |
| `autocorrect_practice` | `practice` returned as dict not list | Fixes silently |
| `validate_required_fields` | Missing top-level or nested fields | Raises `ValidationError` |
| `validate_against_schema` | Content issues (bias, multi-skill, vocab) | Flags, never raises |

**Design decision this validates:**
Structural errors (malformed JSON, missing fields) raise immediately
because there is no safe way to proceed without them.
Content issues (bias, multi-skill) flag but never raise because
a flagged lesson is more useful than no lesson -- it can be reviewed
and corrected by a teacher.

**Known limitation:**
The auto-correction for `practice` as a dict is a workaround for
a specific model behaviour. A production system would use structured
output mode (JSON schema enforcement at the API level) to prevent
this entirely -- Groq supports this via `response_format` parameter.